# Bonus 01 — LangChain foundations for data engineers

LangChain often appears in agent tutorials as one import immediately before the agent starts. That makes it easy to mistake the library for an agent engine.

This notebook starts one layer lower. You will use LangChain as an **integration and composition toolkit** for a familiar data-engineering job:

> turn messy invoice notes into typed records that downstream code can route, store, and test.

There is no agent here. No tool calling. No retrieval. No graph. The program is a fixed pipeline, and that is the right architecture for a fixed transformation.

By the end you should be able to explain:

- what lives in `langchain-core`, `langchain-openai`, `langchain`, and `langgraph`;
- why messages are objects rather than loose strings;
- how a prompt template becomes a reusable data contract;
- how structured output replaces fragile JSON parsing;
- what the `|` operator composes;
- what `.invoke()`, `.batch()`, and their async forms actually do;
- where LangChain helps, and where plain Python remains clearer.

This is an optional bonus lab. It does not add another module to the two-day core course.


## 1. Learn

### The family, before the syntax

| Package | Role | What you use here |
|---|---|---|
| `langchain-core` | Stable interfaces and primitives | messages, prompt templates, runnables |
| `langchain-openai` | The OpenAI adapter | `ChatOpenAI` |
| `langchain` | Higher-level application and agent conveniences | not needed for this fixed pipeline |
| `langgraph` | Explicit state and control flow | not needed until the arrows themselves are the program |

Provider and system integrations live in separate packages. Model adapters, embeddings, vector stores, document loaders, retrievers, tools, checkpointers, and sandboxes can implement LangChain interfaces without all living in one giant package.

That separation is the first lesson: **LangChain is mostly contracts plus adapters.** It is not a model, a database, or a source of business truth.

The course still uses one provider—OpenAI. Knowing that other adapters exist is useful architecture context; hiding provider differences behind a custom abstraction is not part of this lab.

### The pipeline you will build

```mermaid
flowchart LR
    A["Python dict"] --> B["ChatPromptTemplate"]
    B --> C["System + human messages"]
    C --> D["ChatOpenAI adapter"]
    D --> E["Raw AIMessage + token metadata"]
    D --> F["Validated InvoiceTriage object"]
    F --> G["Ordinary Python routing"]
```

The model performs one bounded transformation. It does not choose the next step. Your software owns the sequence from left to right.


### A data-engineering translation

If you work with data pipelines, the concepts already have familiar shapes.

| LangChain word | Data-engineering analogy |
|---|---|
| message | a typed envelope carrying content and metadata |
| prompt template | a parameterised transformation contract |
| model integration | a connector or adapter to an external compute service |
| runnable | a stage with a standard execution interface |
| runnable sequence | a small directed pipeline |
| structured output | a validated record rather than an unparsed text blob |
| `.batch()` | client-side concurrent submission of several records |

The analogy has limits. A model is probabilistic, rate-limited, and billed per token. A successful HTTP response does not mean the extracted facts are correct. You still need validation, evaluation, observability, and a human policy for high-risk records.

### One interface, several execution modes

Most LangChain components implement the `Runnable` interface:

- `invoke(input)` — one input, synchronously;
- `ainvoke(input)` — one input, asynchronously;
- `batch(inputs)` — several inputs, usually with client-side concurrency;
- `abatch(inputs)` — the async batch form;
- `stream(input)` / `astream(input)` — incremental output when every stage supports it.

A `RunnableSequence` is several runnables connected in order. The `|` operator constructs that sequence. It is composition, not hidden planning.


## 2. Do

### Load the environment and the installed interfaces


In [ ]:
from importlib.metadata import version
from pathlib import Path
from typing import Literal
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model_name = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model_name, "MODEL_DEFAULT is missing."

llm = ChatOpenAI(model=model_name, reasoning_effort="none")

print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model_name)
for package in ("langchain", "langchain-core", "langchain-openai"):
    print(f"{package}: {version(package)}")


### Part 1 — Messages are the protocol objects

A chat model does not receive a magical conversation. It receives an ordered list of messages.

LangChain gives those messages standard classes. Each message carries a role, content, and metadata. The provider adapter translates them into the provider's wire format and translates the response back into an `AIMessage`.

Start with one direct call. This is not a chain and not an agent.


In [ ]:
messages = [
    SystemMessage(
        content=(
            "You teach data engineers. Be precise and use one short sentence."
        )
    ),
    HumanMessage(
        content=(
            "What is the practical difference between a model integration "
            "and an agent?"
        )
    ),
]

reply = llm.invoke(messages)

print("input types:", [type(message).__name__ for message in messages])
print("output type:", type(reply).__name__)
print("content:", reply.content)
print("usage:", reply.usage_metadata)


The answer is text, but the returned value is not merely a string. The `AIMessage` also carries token usage, response metadata, IDs, and—when tools are bound—tool-call requests.

That standard message object is why later components can compose. The prompt, model, graph, and agent agree on the envelope.

### Part 2 — A prompt template is a reusable contract

The input to a data pipeline should be a record, not a hand-built f-string scattered through application code.

The prompt below has two inputs: `record_id` and `source`. Formatting it produces messages **without calling a model**. We inspect those messages first.


In [ ]:
records = [
    {
        "record_id": "case-001",
        "source": (
            "Helena Holý says invoice 404 was charged twice. "
            "The disputed amount is $9.90. Please review it."
        ),
    },
    {
        "record_id": "case-002",
        "source": (
            "Puja Srivastava cannot find the invoice PDF for her last order. "
            "She only needs another copy."
        ),
    },
    {
        "record_id": "case-003",
        "source": (
            "Mark says the $3.96 total looks wrong, but the note does not say "
            "which Mark or what the correct amount should be."
        ),
    },
]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "Turn one invoice support note into a typed triage record. "
                "Extract only stated facts. Copy record_id exactly. "
                "Use null when no amount is stated. "
                "needs_human must be true for a duplicate charge, a refund, "
                "an ambiguous identity, or an unclear amount."
            ),
        ),
        (
            "human",
            "record_id: {record_id}\nsource: {source}",
        ),
    ]
)

formatted = prompt.invoke(records[0])

print("prompt input variables:", prompt.input_variables)
for message in formatted.messages:
    print(type(message).__name__, "->", message.content)


No API call happened in that cell. The template transformed a Python dict into the same message objects you built manually in Part 1.

### Part 3 — Ask for a validated record

Free-form JSON in a code fence is still text. It can contain missing keys, commentary, wrong types, or invalid syntax.

`with_structured_output` binds a schema to the model call. We use a Pydantic model because it validates the returned fields at runtime.

`include_raw=True` keeps both sides visible:

- `raw` — the provider response as an `AIMessage`, including token metadata;
- `parsed` — the validated `InvoiceTriage` object;
- `parsing_error` — the failure, if validation did not succeed.

This is an integration boundary, not a truth guarantee. The schema validates shape and types. It does not prove that the model read the note correctly.


In [ ]:
class InvoiceTriage(BaseModel):
    """A typed record produced from one invoice support note."""

    record_id: str = Field(description="Copy the supplied record_id exactly.")
    customer: str = Field(description="Customer name stated in the note.")
    issue_type: Literal[
        "duplicate_charge",
        "missing_invoice",
        "refund_request",
        "amount_question",
        "other",
    ]
    amount_usd: float | None = Field(
        description="The stated US dollar amount, or null when none is stated."
    )
    needs_human: bool = Field(
        description="Whether policy requires a person to review this record."
    )
    summary: str = Field(description="One short factual sentence.")


structured_llm = llm.with_structured_output(
    InvoiceTriage,
    method="json_schema",
    include_raw=True,
)

pipeline = prompt | structured_llm

print("pipeline type:", type(pipeline).__name__)
print("steps:", [type(step).__name__ for step in pipeline.steps])
print("prompt inputs:", prompt.input_variables)


Conceptually, the pipeline has two stages: format messages, then call the structured model adapter. `include_raw=True` expands that second stage into internal capture, parse, and fallback runnables, so the printed implementation list may contain more than two objects. The `|` operator still did not create an autonomous agent. It created a `RunnableSequence`.

### Part 4 — Run several records

`.batch()` accepts a list of inputs and preserves input order in its returned list. Its default implementation uses client-side concurrency for I/O-bound runnables. It is **not** the OpenAI Batch API and it does not remove rate limits.

The concurrency cap matters in production. A pipeline that works for three records can still overload an API or exhaust a budget at three million.


In [ ]:
results = pipeline.batch(
    records,
    config={"max_concurrency": 2},
)

for result in results:
    assert result["parsing_error"] is None
    parsed = result["parsed"]
    print(parsed.model_dump())


## 3. Observe

### Make the invisible boundary visible

Inspect one result all the way through. Then build an ordinary Python review queue from the validated objects.

The model has finished by the time the review policy runs. The final routing decision below is deterministic application code.


In [ ]:
first = results[0]

print("raw type:", type(first["raw"]).__name__)
print("raw content:", repr(first["raw"].content))
print("raw usage:", first["raw"].usage_metadata)
print("parsed type:", type(first["parsed"]).__name__)
print("parsed:", first["parsed"].model_dump())
print("parsing_error:", first["parsing_error"])

review_queue = [
    result["parsed"].model_dump()
    for result in results
    if result["parsed"].needs_human
]

print("\nreview queue:")
for row in review_queue:
    print(row["record_id"], "->", row["issue_type"])


Depending on the provider adapter, raw content may contain JSON text or be empty. Application code should consume `parsed`; keep `raw` for protocol metadata and diagnosis. The important point is that you retained both the provider response and the validated application record.

Now account for the batch.


In [ ]:
input_tokens = sum(
    result["raw"].usage_metadata.get("input_tokens", 0)
    for result in results
)
output_tokens = sum(
    result["raw"].usage_metadata.get("output_tokens", 0)
    for result in results
)

price_in = float(os.environ.get("PRICE_INPUT_PER_MILLION", "0"))
price_out = float(os.environ.get("PRICE_OUTPUT_PER_MILLION", "0"))
batch_cost = (
    input_tokens * price_in / 1_000_000
    + output_tokens * price_out / 1_000_000
)

print("records:", len(results))
print("input_tokens:", input_tokens)
print("output_tokens:", output_tokens)
print(f"estimated batch cost: ${batch_cost:.6f}")


### Inspect the standard interface

The same pipeline object exposes sync, async, batch, and streaming methods. That common interface is what LangChain contributes.

Having a method does not mean every provider implements it identically or that every composed stage can stream immediately. Standard shape and identical semantics are different promises.


In [ ]:
for method_name in (
    "invoke",
    "ainvoke",
    "batch",
    "abatch",
    "stream",
    "astream",
):
    method = getattr(pipeline, method_name)
    print(method_name, "callable:", callable(method))

print("\npipeline graph:")
print(pipeline.get_graph().draw_mermaid())


### What this lab did—and did not—buy

| Concern | Plain provider SDK | LangChain here |
|---|---|---|
| Send messages | explicit provider objects | standard message objects |
| Reuse a prompt | your own function | `ChatPromptTemplate` |
| Validate output | provider schema + your validation | `with_structured_output` + Pydantic |
| Compose stages | ordinary function calls | `RunnableSequence` |
| Submit several inputs | your executor or async code | `.batch()` / `.abatch()` |
| Add an agent loop | you write the loop | deliberately not used here |
| Decide business truth | your code and data | still your code and data |

The honest adoption rule is not “always use a framework” or “never use one.”

Use LangChain when its interfaces and integrations remove real glue across several components. Use the provider SDK and plain Python when the job is one or two transparent calls. Do not wrap a five-line SDK call in a chain merely to say you use LangChain.

Module 09's `create_agent` and LangGraph now have a clearer foundation: they are higher-level programs built on the messages, models, tools, and runnable contracts you used here.


## 4. Challenge

A fourth record arrives:

> Puja Srivastava requests a refund for a $36.64 invoice because the order was cancelled.

Invoke `pipeline` with:

- `record_id`: `case-004`
- the note above as `source`

Bind the complete returned dictionary to `challenge_result`.

Acceptance criteria:

- parsing succeeds;
- the parsed object keeps `case-004`;
- the customer contains Puja's name;
- the issue is `refund_request`;
- the amount is 36.64;
- `needs_human` is true;
- raw token usage is still available.


In [ ]:
# challenge_result = pipeline.invoke(
#     {
#         "record_id": ...,
#         "source": ...,
#     }
# )


In [ ]:
parsed = challenge_result["parsed"]

assert challenge_result["parsing_error"] is None
assert parsed.record_id == "case-004"
assert "puja" in parsed.customer.lower()
assert parsed.issue_type == "refund_request"
assert abs(parsed.amount_usd - 36.64) < 0.001
assert parsed.needs_human is True
assert challenge_result["raw"].usage_metadata.get("total_tokens", 0) > 0

print(parsed.model_dump())
